# Perplexity Analysis — TAT German

This notebook runs the full statistical analysis and visualization pipeline.
It loops over three analysis settings (with outliers, without outliers, matched sample)
and produces the same set of plots and statistics for each.

In [1]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
import matplotlib
matplotlib.use('Agg')

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Make sure project root is on the path
sys.path.insert(0, os.path.abspath('.'))

from services.analysis_service import analysis_service, load_metadata, build_person_df
from services.plotting_service import plot_setting, plot_cross_setting, plot_advanced, plot_matched_sample_balance
from functions.analysis.statistics import (
    CLINICAL_VARS,
    group_comparison,
    clinical_correlations,
    residualize_ppl,
    nested_regression_models,
    steigers_z,
    compute_roc,
    logistic_cv_auc,
)
from functions.plotting.plots import (
    plot_group_boxplot,
    plot_roc_curves,
    plot_metrics_heatmap,
    plot_raw_vs_residualized,
    format_and_save_clinical_correlations,
)

## 1. Configuration

In [2]:
# Paths
TYPE = "prompt_healthy" #prompt_neutral prompt_no_prompt prompt_healthy prompt_sick
SENTENCE_CSV = f"/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/data/02_perplexities/Qwen2.5-32B-Instruct/{TYPE}/TAT_german_utterances_with_perplexities_{TYPE.split('_', 1)[1]}.csv"
METADATA_CSV = "data/00_raw_data/speech_metadata.csv"
FIGURE_DIR = "figures"
METADATA_COLS = [
    "file_id", "diagnosis", "sex", "age", "education", "comorbidity",
    "GAF", "verbal_IQ", "TLI_Impoverished", "TLI_Disorganization", "TLI_Non_specific",
]

# Model prefix for column names
MODEL = "qwen"

# Primary perplexity variable
PPL_VAR = f"{MODEL}_ppl_median"
WC_VAR = f"{MODEL}_total_words"

# Clinical variable setup: small set for plots, larger set for correlation tables
extra_clinical_vars = [
    # Add any additional metadata columns you want to test here, e.g. "PANSS_Total"
    # "PANSS_Total",
    # "CDSS_Total",
]

# Keep full candidate list here; we will validate against df.columns after df is built
clinical_vars_corr = list(dict.fromkeys(CLINICAL_VARS + extra_clinical_vars))
clinical_vars = [v for v in ["GAF", "TLI_Impoverished"] if v in clinical_vars_corr]

print(f"Clinical vars for plots (requested): {clinical_vars}")
print(f"Clinical vars for correlation tables (requested, pre-validation): {len(clinical_vars_corr)}")

Clinical vars for plots (requested): ['GAF', 'TLI_Impoverished']
Clinical vars for correlation tables (requested, pre-validation): 19


## 2. Load data & prepare three analysis settings

In [3]:
match_vars = [f"{MODEL}_avg_words", "education"]
CONFOUNDER_VARS = ["sex", "age", f"{MODEL}_avg_words", "education"] #"verbal_IQ"
MATCH_K = 2  # number of HCs per patient in per-diagnosis matching
df, settings = analysis_service(
    sentence_csv=SENTENCE_CSV,
    metadata_csv=METADATA_CSV,
    model_name=MODEL,
    match_vars=match_vars,
    match_k=MATCH_K,
    requested_clinical_vars=clinical_vars_corr,
    impute_confounders_cols=CONFOUNDER_VARS,
 )

# Validate requested clinical vars against the actual analysis df (after rename + merge)
missing_corr_vars = [v for v in clinical_vars_corr if v not in df.columns]
clinical_vars_corr = [v for v in clinical_vars_corr if v in df.columns]
clinical_vars = [v for v in clinical_vars if v in clinical_vars_corr]

print(f"\nFull dataset: {len(df)} subjects")
print(f"Patients: {(df['is_patient'] == 1).sum()}, Controls: {(df['is_patient'] == 0).sum()}")
print(f"Clinical vars for correlation tables (available): {len(clinical_vars_corr)}")
if missing_corr_vars:
    print(f"Dropped missing clinical vars: {missing_corr_vars}")

FileNotFoundError: [Errno 2] No such file or directory: '/zi/home/esra.lenz/Documents/00_HITKIP/11_normative/mnm_ppl/data/02_perplexities/Qwen2.5-32B-Instruct/prompt_healthy/TAT_german_utterances_with_perplexities_healthy.csv'

In [ ]:
# ── Diagnostic: which variables actually correlate with PPL? ──────
from scipy.stats import spearmanr as _sp, mannwhitneyu
def fmt_p(p):
    if p < 0.001:
        return "< 0.001"
    elif p < 0.01:
        return f"{p:.3f}"
    else:
        return f"{p:.2f}"
candidate_vars = ["sex", "age", "education", "verbal_IQ", f"{MODEL}_avg_words"]
print(f"{'Variable':<25} {'r_all':>8} {'p_all':>10} {'r_HC':>8} {'p_HC':>10} {'r_PAT':>8} {'p_PAT':>10}")
print("-" * 85)

for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1}).fillna(0.5)
    else:
        col = df[var]

    # All subjects
    mask_all = col.notna() & df[PPL_VAR].notna()
    r_all, p_all = _sp(col[mask_all], df[PPL_VAR][mask_all])

    # HC only
    hc = df["is_patient"] == 0
    mask_hc = hc & col.notna() & df[PPL_VAR].notna()
    r_hc, p_hc = _sp(col[mask_hc], df[PPL_VAR][mask_hc])

    # Patients only
    pat = df["is_patient"] == 1
    mask_pat = pat & col.notna() & df[PPL_VAR].notna()
    r_pat, p_pat = _sp(col[mask_pat], df[PPL_VAR][mask_pat])

    sig_all = "***" if p_all < .001 else "**" if p_all < .01 else "*" if p_all < .05 else "ns"
    sig_hc = "***" if p_hc < .001 else "**" if p_hc < .01 else "*" if p_hc < .05 else "ns"
    sig_pat = "***" if p_pat < .001 else "**" if p_pat < .01 else "*" if p_pat < .05 else "ns"

    print(f"{var:<25} {r_all:>+.3f} {p_all:>10.4f} {sig_all:>3}  {r_hc:>+.3f} {p_hc:>10.4f} {sig_hc:>3}  {r_pat:>+.3f} {p_pat:>10.4f} {sig_pat:>3}")

# Also check: do these variables differ between groups?
print(f"\n{'Variable':<25} {'HC mean':>10} {'PAT mean':>10} {'p(MW)':>10} {'sig':>4}")
print("-" * 65)
for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1}).fillna(0.5)
    else:
        col = df[var]
    hc_vals = col[df["is_patient"] == 0].dropna()
    pat_vals = col[df["is_patient"] == 1].dropna()
    stat, p_mw = mannwhitneyu(hc_vals, pat_vals, alternative="two-sided")
    sig = "***" if p_mw < .001 else "**" if p_mw < .01 else "*" if p_mw < .05 else "ns"
    print(f"{var:<25} {hc_vals.mean():>10.3f} {pat_vals.mean():>10.3f} {p_mw:>10.4f} {sig:>4}")

import pandas as pd

# ── Table 1: Spearman correlations with PPL ──────
rows_corr = []
for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1}).fillna(0.5)
    else:
        col = df[var]

    mask_all = col.notna() & df[PPL_VAR].notna()
    r_all, p_all = _sp(col[mask_all], df[PPL_VAR][mask_all])

    hc = df["is_patient"] == 0
    mask_hc = hc & col.notna() & df[PPL_VAR].notna()
    r_hc, p_hc = _sp(col[mask_hc], df[PPL_VAR][mask_hc])

    pat = df["is_patient"] == 1
    mask_pat = pat & col.notna() & df[PPL_VAR].notna()
    r_pat, p_pat = _sp(col[mask_pat], df[PPL_VAR][mask_pat])

    rows_corr.append({
        "Variable": var,
        "ρ_all": round(r_all, 2), "P_all": fmt_p(p_all),
        "ρ_HC": round(r_hc, 2),  "P_HC": fmt_p(p_hc),
        "ρ_PAT": round(r_pat, 2), "P_PAT": fmt_p(p_pat),
    })

df_corr = pd.DataFrame(rows_corr)

# ── Table 2: Group comparisons (Mann-Whitney U) ──────
rows_mw = []
for var in candidate_vars:
    if var == "sex":
        col = df["sex"].map({"M": 0, "F": 1, "m": 0, "f": 1, "male": 0, "female": 1, "Male": 0, "Female": 1}).fillna(0.5)
    else:
        col = df[var]
    hc_vals = col[df["is_patient"] == 0].dropna()
    pat_vals = col[df["is_patient"] == 1].dropna()
    stat, p_mw = mannwhitneyu(hc_vals, pat_vals, alternative="two-sided")

    rows_mw.append({
        "Variable": var,
        "HC mean": round(hc_vals.mean(), 2),
        "HC SD": round(hc_vals.std(), 2),
        "PAT mean": round(pat_vals.mean(), 2),
        "PAT SD": round(pat_vals.std(), 2),
        "U": round(stat, 1),
        "P": fmt_p(p_mw),
    })

df_mw = pd.DataFrame(rows_mw)

# ── Save both to one Excel file with two sheets ──────
with pd.ExcelWriter(f"{FIGURE_DIR}/confounder/supplementary_confounder_tables.xlsx") as writer:
    df_corr.to_excel(writer, sheet_name="Spearman_correlations", index=False)
    df_mw.to_excel(writer, sheet_name="Group_comparisons", index=False)

print("Saved: supplementary_confounder_tables.xlsx")

In [ ]:
# ── Correlation: matching variables & confounders vs perplexity ────────────
from scipy.stats import spearmanr as _sp

plot_vars = match_vars + CONFOUNDER_VARS
plot_vars = [v if v != "sex" else "sex_num" for v in match_vars + CONFOUNDER_VARS]
nice_names = {
    f"{MODEL}_avg_words": "Avg words per sentence",
    "sex": "Sex (0=m, 1=f)",
    "age": "Age",
}

n_vars = len(plot_vars)
fig, axes = plt.subplots(1, n_vars, figsize=(5.5 * n_vars, 4.5))
if n_vars == 1:
    axes = [axes]

for ax, var in zip(axes, plot_vars):
    sub = df[[var, PPL_VAR]].dropna()
    is_cat = sub[var].nunique() <= 5
    x_vals = sub[var] + (np.random.uniform(-0.15, 0.15, len(sub)) if is_cat else 0)
    ax.scatter(x_vals, sub[PPL_VAR], alpha=0.45, s=18, color="#5B7DB1", edgecolors="none")
    # regression line + Spearman r
    r, p = _sp(sub[var], sub[PPL_VAR])
    z = np.polyfit(sub[var].astype(float), sub[PPL_VAR], 1)
    xs = np.linspace(sub[var].min(), sub[var].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", lw=1.5, alpha=0.7)
    ax.set_xlabel(nice_names.get(var, var), fontsize=11)
    ax.set_ylabel("Median perplexity" if ax == axes[0] else "")
    sig = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
    ax.set_title(f"r = {r:.3f}, p = {p:.4f} {sig}", fontsize=10)
    ax.grid(alpha=0.2)

fig.suptitle(f"Matching & confounder variables vs {PPL_VAR}", fontsize=13, y=1.02)
fig.tight_layout()
os.makedirs(os.path.join(FIGURE_DIR, "overview"), exist_ok=True)
fig.savefig(os.path.join(FIGURE_DIR, "overview", "confounder_ppl_correlations.png"),
            dpi=150, bbox_inches="tight")
plt.show()

### Matching quality: confounder balance (before vs after matching)

In [ ]:
plot_matched_sample_balance(df, settings["matched"], match_vars, FIGURE_DIR)
print("\n--- Per-diagnosis 1:k matched ---")
plot_matched_sample_balance(df, settings["matched_per_dx"], match_vars, FIGURE_DIR)

## 3. Loop: Group comparison & clinical correlations for all 3 settings

In [ ]:
import statsmodels.api as sm

# Map each setting to the PPL variable it should use
setting_ppl_vars = {
    "with_outliers": PPL_VAR,
    "without_outliers": f"{PPL_VAR}_wins",
    "matched": PPL_VAR,
    "matched_without_outliers": f"{PPL_VAR}_wins",
    "matched_per_dx": PPL_VAR,
    "matched_per_dx_without_outliers": f"{PPL_VAR}_wins",
}

all_results = {}
all_corr_dfs = {}

for setting_name, setting_df in settings.items():
    ppl_col = setting_ppl_vars[setting_name]
    print(f"\n{'='*60}")
    print(f"Setting: {setting_name} | PPL variable: {ppl_col}")
    print(f"{'='*60}")

    # ── Standard analysis ─────────────────────────────────────────
    result = plot_setting(
        df=setting_df,
        ppl_var=ppl_col,
        setting_name=setting_name,
        figure_dir=FIGURE_DIR,
        clinical_vars=clinical_vars,
    )

    # ── Deconfounding analysis ────────────────────────────────────
    print(f"\n--- Deconfounding ({setting_name}) ---")
    sdf = setting_df.copy()
    sdf = sdf.reset_index(drop=True)
    sdf["log_words"] = sdf[WC_VAR]
    confounders = CONFOUNDER_VARS

    resid = residualize_ppl(sdf, ppl_col, confounders)
    resid_col = f"{ppl_col}_resid"
    sdf[resid_col] = np.nan
    sdf.loc[resid.index, resid_col] = resid

    # Group comparison: raw vs residualized
    comp_raw = group_comparison(sdf, ppl_col)
    comp_resid = group_comparison(sdf.dropna(subset=[resid_col]), resid_col)
    print(f"  Raw:          p(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    print(f"  Residualized: p(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")

    # Clinical correlations: raw vs residualized
    patients_s = sdf[sdf["is_patient"] == 1].copy()
    corr_raw = clinical_correlations(patients_s, ppl_col, clinical_vars_corr)
    corr_resid = clinical_correlations(
        patients_s.dropna(subset=[resid_col]), resid_col, clinical_vars_corr
    )
    merged_corr = corr_raw.merge(
        corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
    )

    # Format, clean, and save clinical correlation tables
    merged_corr = format_and_save_clinical_correlations(
        merged_corr, setting_name, FIGURE_DIR, fmt_p
    )

    # Pairwise vs HC: raw vs residualized
    from functions.analysis.statistics import pairwise_vs_hc, pairwise_vs_matched_hc, kruskal_wallis_test
    is_matched = "match_id" in sdf.columns
    pw_func = pairwise_vs_matched_hc if is_matched else pairwise_vs_hc
    pw_raw = pw_func(sdf, ppl_col)
    pw_resid = pw_func(sdf.dropna(subset=[resid_col]), resid_col)
    kw_raw_stat, kw_raw_p = kruskal_wallis_test(sdf, ppl_col)
    kw_res_stat, kw_res_p = kruskal_wallis_test(
        sdf.dropna(subset=[resid_col]), resid_col
    )
    print(f"\n  Kruskal-Wallis — Raw: H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
    print(f"  Kruskal-Wallis — Resid: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

    # ── Per-diagnosis table: each diagnosis vs HC ─────────────────
    pw_raw_df = pd.DataFrame(pw_raw)
    pw_res_df = pd.DataFrame(pw_resid)
    pw_merged = pw_raw_df[["Diagnosis", "n"]].copy()
    if "n(HC)" in pw_raw_df.columns:
        pw_merged.insert(1, "n(HC)", pw_raw_df["n(HC)"])
    else:
        pw_merged.insert(1, "n(HC)", len(sdf[sdf["diagnosis"] == "HC"]))
    pw_merged["d (raw)"] = pw_raw_df["Cohen's d"]
    pw_merged["p (raw)"] = pw_raw_df["p"]
    pw_merged["d (resid)"] = pw_res_df["Cohen's d"]
    pw_merged["p (resid)"] = pw_res_df["p"]
    pw_merged["sig (raw)"] = pw_merged["p (raw)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    pw_merged["sig (resid)"] = pw_merged["p (resid)"].apply(
        lambda x: "***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "ns"
    )
    print(f"\n  Per-diagnosis vs HC — {setting_name}:")
    display(pw_merged.style
        .format({"d (raw)": "{:.3f}", "p (raw)": "{:.4f}",
                 "d (resid)": "{:.3f}", "p (resid)": "{:.4f}"})
        .set_caption(f"Diagnosis vs HC — {setting_name}")
    )

    # ── Deconfounded visualisation ────────────────────────────────
    deconf_dir = os.path.join(FIGURE_DIR, setting_name, "deconfounded")
    os.makedirs(deconf_dir, exist_ok=True)

    # Plot 1: raw vs residualized (uses plotting_service helper)
    plot_raw_vs_residualized(
        sdf, ppl_col, WC_VAR,
        confounders=CONFOUNDER_VARS,
        save_path=os.path.join(deconf_dir, "raw_vs_residualized.png"),
    )

    # Plot 1b: HC vs Patient boxplot after deconfounding
    plot_group_boxplot(
        sdf.dropna(subset=[resid_col]), resid_col,
        title=f"{setting_name}: HC vs Patient (deconfounded)",
        save_path=os.path.join(deconf_dir, "group_boxplot_deconfounded.png"),
        ylabel="Median PPL",
        plot_style="raincloud"
    )

    # Plot 2: deconfounded comparison figure
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    from scipy.stats import spearmanr as _spearmanr

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Row 1 col 0: Raw PPL violin
    ax = axes[0, 0]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][ppl_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Raw {ppl_col}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
    ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

    # Row 1 col 1: Residualized PPL violin
    ax = axes[0, 1]
    for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
        vals = sdf[sdf["is_patient"] == is_pat][resid_col].dropna()
        parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
        for pc in parts["bodies"]:
            pc.set_facecolor(clr); pc.set_alpha(0.6)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
    ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
    ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

    # Row 1 col 2: Cohen's d comparison
    ax = axes[0, 2]
    ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
           color=["#3498DB", "#E74C3C"], alpha=0.8)
    ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
    ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

    # Row 2 col 0: Clinical correlations bar chart (plot subset only)
    ax = axes[1, 0]
    x = np.arange(len(clinical_vars))
    w = 0.35
    bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
    bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
    ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
    ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
    ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
    ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

    # Row 2 col 1: Scatter — residualized PPL vs TLI Impoverished
    ax = axes[1, 1]
    sub = patients_s[[resid_col, "TLI_Impoverished", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["TLI_Impoverished"])
        z = np.polyfit(sub[resid_col], sub["TLI_Impoverished"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
        ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    # Row 2 col 2: Scatter — residualized PPL vs GAF
    ax = axes[1, 2]
    sub = patients_s[[resid_col, "GAF", "diagnosis"]].dropna()
    if len(sub) >= 5:
        diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
        for dx in diags:
            s = sub[sub["diagnosis"] == dx]
            ax.scatter(s[resid_col], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                       label=dx, alpha=0.6, s=25)
        r, p = _spearmanr(sub[resid_col], sub["GAF"])
        z = np.polyfit(sub[resid_col], sub["GAF"], 1)
        xs = np.linspace(sub[resid_col].min(), sub[resid_col].max(), 100)
        ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
        ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
        ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
        ax.legend(fontsize=7); ax.grid(alpha=0.2)

    fig.suptitle(f"Deconfounded analysis — {setting_name}", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(deconf_dir, "deconfounded_analysis.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

    # ── Store results ─────────────────────────────────────────────
    result["deconfounded"] = {
        "comparison_raw": comp_raw,
        "comparison_resid": comp_resid,
        "correlations_raw": corr_raw,
        "correlations_resid": corr_resid,
        "merged_correlations": merged_corr,
        "pairwise_raw": pw_raw,
        "pairwise_resid": pw_resid,
        "kruskal_raw": (kw_raw_stat, kw_raw_p),
        "kruskal_resid": (kw_res_stat, kw_res_p),
    }
    all_results[setting_name] = result
    all_corr_dfs[setting_name] = result["correlations"]

## 4. Cross-setting comparison

In [ ]:
plot_cross_setting(all_corr_dfs, figure_dir=FIGURE_DIR)

In [ ]:
# Summary table of all comparisons (raw + deconfounded)
summary_rows = []
for sname, res in all_results.items():
    row_raw = {"Setting": sname, "Analysis": "Raw"}
    row_raw.update(res["comparison"])
    summary_rows.append(row_raw)
    if "deconfounded" in res:
        row_resid = {"Setting": sname, "Analysis": "Residualized"}
        row_resid.update(res["deconfounded"]["comparison_resid"])
        summary_rows.append(row_resid)
summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. Alternative metrics comparison

In [ ]:
alt_metrics = [
    f"{MODEL}_ppl_median", f"{MODEL}_ppl_mean",
    f"{MODEL}_ppl_p90", f"{MODEL}_ppl_p75",
    f"{MODEL}_ppl_std", f"{MODEL}_ppl_cv",
    f"{MODEL}_prop_high_ppl",
]

# Group comparison for each metric
alt_comp = pd.DataFrame([group_comparison(df, m) for m in alt_metrics if m in df.columns])
alt_comp

In [ ]:
# Correlation heatmap: metrics × clinical variables
patients = df[df['is_patient'] == 1].copy()
r_matrix = pd.DataFrame(index=[m for m in alt_metrics if m in df.columns], columns=clinical_vars, dtype=float)
p_matrix = r_matrix.copy()

for metric in r_matrix.index:
    for cvar in clinical_vars:
        sub = patients[[metric, cvar]].dropna()
        if len(sub) >= 5:
            r, p = spearmanr(sub[metric], sub[cvar])
            r_matrix.loc[metric, cvar] = r
            p_matrix.loc[metric, cvar] = p

plot_metrics_heatmap(
    r_matrix.astype(float),
    p_matrix.astype(float),
    title="Alternative metrics × Clinical variables (Spearman r)",
    save_path=os.path.join(FIGURE_DIR, "advanced", "metrics_heatmap.png"),
)

## 6. TLI subscale analysis (Steiger's Z)

In [ ]:
tli_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
tli_vars = ["TLI_Impoverished", "TLI_Disorganization", "TLI_Non_specific"]

for metric in tli_metrics:
    sub = patients[[metric] + tli_vars].dropna()
    if len(sub) < 10:
        continue
    r_imp, _ = spearmanr(sub[metric], sub["TLI_Impoverished"])
    r_dis, _ = spearmanr(sub[metric], sub["TLI_Disorganization"])
    r_id, _ = spearmanr(sub["TLI_Impoverished"], sub["TLI_Disorganization"])
    z, p = steigers_z(r_imp, r_dis, r_id, len(sub))
    print(f"{metric}: r_imp={r_imp:.3f}, r_dis={r_dis:.3f}, Steiger Z={z:.3f}, p={p:.4f}")

## 7. ROC analysis

In [ ]:
# Define clinical impairment targets
patients = df[df['is_patient'] == 1].copy()
patients['GAF_impaired'] = (patients['GAF'] < 60).astype(int)
patients['TLI_dis_any'] = (patients['TLI_Disorganization'] > 0).astype(int)
patients['TLI_imp_any'] = (patients['TLI_Impoverished'] > 0).astype(int)

ppl_roc_metrics = [PPL_VAR, f"{MODEL}_ppl_p90", f"{MODEL}_prop_high_ppl"]
roc_targets = {
    'GAF < 60': 'GAF_impaired',
    'TLI Disorganization > 0': 'TLI_dis_any',
    'TLI Impoverished > 0': 'TLI_imp_any',
}

all_rocs = {}
for target_label, target_col in roc_targets.items():
    rocs = compute_roc(patients, ppl_roc_metrics, target_col)
    all_rocs[target_label] = rocs
    for r in rocs:
        print(f"  {target_label} | {r['metric']}: AUC={r['auc']:.3f}")

plot_roc_curves(
    [],  # not used directly
    all_rocs,
    save_path=os.path.join(FIGURE_DIR, "advanced", "roc_curves.png"),
)

In [ ]:
# Combined metric: logistic regression CV
features_combined = [PPL_VAR, f"{MODEL}_ppl_std", f"{MODEL}_prop_high_ppl"]
for target_label, target_col in roc_targets.items():
    auc_combined = logistic_cv_auc(patients, features_combined, target_col)
    auc_singles = {m: logistic_cv_auc(patients, [m], target_col) for m in features_combined}
    print(f"{target_label}: combined AUC={auc_combined:.3f}")
    for m, a in auc_singles.items():
        print(f"  single {m}: AUC={a:.3f}")

## 8. Confounder analysis

In [ ]:
# Confounder visualisation
plot_advanced(
    df=df,
    patients=patients,
    ppl_var=PPL_VAR,
    wc_var=WC_VAR,
    figure_dir=FIGURE_DIR,
)

### 8b. Deconfounding: residualize PPL, then re-run group comparisons & clinical correlations

Instead of nested ANOVA (where confounders absorb the group signal), we regress out
confounders (log word count + education) from perplexity and then test the **residualized**
perplexity for group differences and clinical associations.

In [ ]:
# ── Residualize perplexity ────────────────────────────────────────
# Regress out log(word_count) + education from PPL, keep the residuals.
import statsmodels.api as sm

df_deconf = df.copy()
df_deconf["log_words"] = np.log1p(df_deconf[WC_VAR])

confounders = CONFOUNDER_VARS

df_deconf[f"{PPL_VAR}_resid"] = np.nan
resid = residualize_ppl(df_deconf, PPL_VAR, confounders)
df_deconf.loc[resid.index, f"{PPL_VAR}_resid"] = resid

RESID_VAR = f"{PPL_VAR}_resid"
print(f"Confounders removed: {confounders}")
print(f"Residualized variable: {RESID_VAR}")
print(f"N with residuals: {df_deconf[RESID_VAR].notna().sum()}")

# ── Group comparison: raw vs residualized ────────────────────────
print("\n" + "=" * 60)
print("GROUP COMPARISON — Raw PPL")
print("=" * 60)
comp_raw = group_comparison(df_deconf, PPL_VAR)
for k, v in comp_raw.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("GROUP COMPARISON — Residualized PPL")
print("=" * 60)
comp_resid = group_comparison(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
for k, v in comp_resid.items():
    print(f"  {k}: {v}")

# Summary table
comp_summary = pd.DataFrame([
    {"Metric": "Raw PPL", **{k: v for k, v in comp_raw.items() if k != "Variable"}},
    {"Metric": "Residualized PPL", **{k: v for k, v in comp_resid.items() if k != "Variable"}},
])
comp_summary

In [ ]:
# ── Clinical correlations: raw vs residualized ───────────────────
patients_deconf = df_deconf[df_deconf["is_patient"] == 1].copy()

print("CLINICAL CORRELATIONS — Raw PPL (patients only)")
corr_raw = clinical_correlations(patients_deconf, PPL_VAR, clinical_vars_corr)
print(corr_raw.to_string(index=False))

print("\nCLINICAL CORRELATIONS — Residualized PPL (patients only)")
corr_resid = clinical_correlations(
    patients_deconf.dropna(subset=[RESID_VAR]), RESID_VAR, clinical_vars_corr
)
print(corr_resid.to_string(index=False))

# Side-by-side comparison
merged_corr = corr_raw.merge(
    corr_resid, on="Clinical Variable", suffixes=("_raw", "_resid")
)
merged_corr

In [ ]:
# ── Visualisation: raw vs residualized ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Raw PPL
ax = axes[0, 0]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][PPL_VAR].dropna()
    ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Raw {PPL_VAR}\np(MW)={comp_raw['p(MW)']:.4f}, d={comp_raw['Cohens d']:.3f}")
ax.set_ylabel("PPL"); ax.grid(alpha=0.3)

# Row 1: Residualized PPL
ax = axes[0, 1]
for is_pat, lbl, clr in [(0, "HC", "#3498DB"), (1, "Patient", "#E74C3C")]:
    vals = df_deconf[df_deconf["is_patient"] == is_pat][RESID_VAR].dropna()
    parts = ax.violinplot([vals], positions=[is_pat], showmeans=True, showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(clr); pc.set_alpha(0.6)
ax.set_xticks([0, 1]); ax.set_xticklabels(["HC", "Patient"])
ax.set_title(f"Residualized PPL\np(MW)={comp_resid['p(MW)']:.4f}, d={comp_resid['Cohens d']:.3f}")
ax.set_ylabel("Residual PPL"); ax.grid(alpha=0.3)

# Row 1: Cohen's d comparison
ax = axes[0, 2]
ax.bar(["Raw", "Residualized"], [comp_raw["Cohens d"], comp_resid["Cohens d"]],
       color=["#3498DB", "#E74C3C"], alpha=0.8)
ax.set_ylabel("Cohen's d"); ax.set_title("Effect size: HC vs Patient")
ax.grid(axis="y", alpha=0.3); ax.axhline(0, color="black", lw=0.5)

# Row 2: Clinical correlations comparison
ax = axes[1, 0]
x = np.arange(len(clinical_vars))
w = 0.35
bars_raw = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_raw"]
bars_res = merged_corr.set_index("Clinical Variable").reindex(clinical_vars)["r_resid"]
ax.bar(x - w/2, bars_raw, w, label="Raw", color="#3498DB", alpha=0.8)
ax.bar(x + w/2, bars_res, w, label="Residualized", color="#E74C3C", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(clinical_vars, rotation=30, ha="right")
ax.set_ylabel("Spearman r"); ax.set_title("Clinical correlations (patients)")
ax.legend(); ax.axhline(0, color="black", lw=0.5); ax.grid(axis="y", alpha=0.3)

# Row 2: Scatter — residualized PPL vs TLI Impoverished
ax = axes[1, 1]
sub = patients_deconf[[RESID_VAR, "TLI_Impoverished", "diagnosis"]].dropna()
if len(sub) >= 5:
    from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS
    diags = [d for d in DIAG_ORDER if d in sub["diagnosis"].values]
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["TLI_Impoverished"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["TLI_Impoverished"])
    z = np.polyfit(sub[RESID_VAR], sub["TLI_Impoverished"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("TLI Impoverished")
    ax.set_title(f"Resid PPL vs TLI Imp (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

# Row 2: Scatter — residualized PPL vs GAF
ax = axes[1, 2]
sub = patients_deconf[[RESID_VAR, "GAF", "diagnosis"]].dropna()
if len(sub) >= 5:
    for dx in diags:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[RESID_VAR], s["GAF"], color=DIAG_COLORS.get(dx, "#888"),
                   label=dx, alpha=0.6, s=25)
    r, p = spearmanr(sub[RESID_VAR], sub["GAF"])
    z = np.polyfit(sub[RESID_VAR], sub["GAF"], 1)
    xs = np.linspace(sub[RESID_VAR].min(), sub[RESID_VAR].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6)
    ax.set_xlabel("Residualized PPL"); ax.set_ylabel("GAF")
    ax.set_title(f"Resid PPL vs GAF (r={r:.3f}, p={p:.4f})")
    ax.legend(); ax.grid(alpha=0.2)

fig.suptitle("Deconfounded analysis: raw vs residualized perplexity", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "advanced", "deconfounded_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-diagnosis deconfounded comparison vs HC ──────────────────
from functions.analysis.statistics import pairwise_vs_hc, kruskal_wallis_test

print("PAIRWISE vs HC — Raw PPL")
pw_raw = pairwise_vs_hc(df_deconf, PPL_VAR)
print(pd.DataFrame(pw_raw).to_string(index=False))

print("\nPAIRWISE vs HC — Residualized PPL")
pw_resid = pairwise_vs_hc(df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR)
print(pd.DataFrame(pw_resid).to_string(index=False))

print("\nKruskal-Wallis across diagnoses:")
kw_raw_stat, kw_raw_p = kruskal_wallis_test(df_deconf, PPL_VAR)
kw_res_stat, kw_res_p = kruskal_wallis_test(
    df_deconf.dropna(subset=[RESID_VAR]), RESID_VAR
)
print(f"  Raw:          H={kw_raw_stat:.3f}, p={kw_raw_p:.4f}")
print(f"  Residualized: H={kw_res_stat:.3f}, p={kw_res_p:.4f}")

## Figure 2: Residualized PPL vs YMRS & SAPS (without_outliers)

In [ ]:
# ── Figure 2: Residualized PPL vs YMRS_Sum & SAPS_Sum_ENIGMA ─────
# Uses the "without_outliers" setting
from scipy.stats import spearmanr as _spearmanr
from functions.plotting.plots import DIAG_ORDER, DIAG_COLORS

# Reconstruct the residualized data for without_outliers
wo_df = settings["without_outliers"].copy().reset_index(drop=True)
wo_ppl = setting_ppl_vars["without_outliers"]
wo_resid_col = f"{wo_ppl}_resid"
wo_df["log_words"] = wo_df[WC_VAR]
wo_resid = residualize_ppl(wo_df, wo_ppl, CONFOUNDER_VARS)
wo_df[wo_resid_col] = np.nan
wo_df.loc[wo_resid.index, wo_resid_col] = wo_resid
wo_patients = wo_df[wo_df["is_patient"] == 1].copy()

clinical_targets = ["YMRS_Sum", "SAPS_Sum_ENIGMA"]
nice = {"YMRS_Sum": "YMRS Sum", "SAPS_Sum_ENIGMA": "SAPS Sum"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, cvar in enumerate(clinical_targets):
    ax = axes[i]
    sub = wo_patients[[wo_resid_col, cvar, "diagnosis"]].dropna()
    if len(sub) < 5:
        ax.text(0.5, 0.5, f"Not enough data for {cvar}", transform=ax.transAxes, ha="center")
        continue

    # Scatter by diagnosis
    for dx in [d for d in DIAG_ORDER if d != "HC" and d in sub["diagnosis"].values]:
        s = sub[sub["diagnosis"] == dx]
        ax.scatter(s[wo_resid_col], s[cvar],
                   color=DIAG_COLORS.get(dx, "#888"), label=dx,
                   alpha=0.65, s=35, edgecolors="white", linewidths=0.3)

    # Regression line + stats (across all patients)
    r, p = _spearmanr(sub[wo_resid_col], sub[cvar])
    z = np.polyfit(sub[wo_resid_col], sub[cvar], 1)
    xs = np.linspace(sub[wo_resid_col].min(), sub[wo_resid_col].max(), 100)
    ax.plot(xs, np.polyval(z, xs), "k--", alpha=0.6, linewidth=1.2)

    # Labels
    p_str = f"p={p:.4f}" if p >= 0.0001 else f"p={p:.2e}"
    ax.set_xlabel("Median PPL (residualized)", fontsize=11)
    ax.set_ylabel(nice[cvar], fontsize=11)
    ax.set_title(f"{nice[cvar]}  (r={r:.3f}, {p_str})", fontsize=12)
    ax.legend(fontsize=8, framealpha=0.8)
    ax.grid(alpha=0.2)

fig.suptitle("Residualized Perplexity vs Clinical Scales (without outliers)",
             fontsize=13, y=1.02)
fig.tight_layout()

save_path = os.path.join(FIGURE_DIR, "main_figures", "fig2_ymrs_saps_scatter.png")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved to {save_path}")
plt.show()

## 9. Summary

In [ ]:
print("Analysis complete.")
print(f"Figures saved to: {os.path.abspath(FIGURE_DIR)}")